# Conciliación de pólizas: sistema vs. aseguradora

Ejecuta cada celda con **Shift + Enter**, en orden.

## 0. Obtener los datos

Si abriste este cuaderno desde GitHub, la celda clona el repositorio para tener la carpeta `datos/`. Cambia `TU_USUARIO` por tu usuario de GitHub. Si el clon falla, te pedirá subir los CSV manualmente.

In [ ]:
import os

REPO = "https://github.com/TU_USUARIO/conciliacion-polizas.git"

if not os.path.exists("datos/sistema.csv"):
    os.system(f"git clone {REPO} repo && cp -r repo/datos .")

if not os.path.exists("datos/sistema.csv"):
    from google.colab import files
    print("Sube sistema.csv y aseguradora.csv")
    subidos = files.upload()
    os.makedirs("datos", exist_ok=True)
    for nombre in subidos:
        destino = "datos/sistema.csv" if "sistema" in nombre else "datos/aseguradora.csv"
        os.replace(nombre, destino)

print(os.listdir("datos"))

## 1. Cargar y limpiar

`utf-8-sig` quita un carácter invisible (BOM) del inicio de los CSV. `errors="coerce"` convierte "N/A" y vacíos a NaN para no inventar montos.

In [ ]:
import pandas as pd

s = pd.read_csv("datos/sistema.csv", encoding="utf-8-sig")
a = pd.read_csv("datos/aseguradora.csv", encoding="utf-8-sig")
s["prima"] = pd.to_numeric(s["prima"], errors="coerce")
a["prima"] = pd.to_numeric(a["prima"], errors="coerce")

## Nivel 1: Pólizas activas

In [ ]:
s[s["estatus"] == "Activa"]

## Nivel 2: Comparar ambas fuentes

`indicator=True` agrega la columna `_merge`: `left_only` = solo en sistema, `right_only` = solo en aseguradora, `both` = en ambas.

In [ ]:
m = s.merge(a, on="id_poliza", how="outer",
            suffixes=("_sistema", "_aseguradora"), indicator=True)

print("(a) En sistema pero NO en aseguradora:")
print(m[m["_merge"] == "left_only"][["id_poliza", "nombre_asegurado_sistema"]].to_string(index=False))

print("\n(b) En aseguradora pero NO en sistema:")
print(m[m["_merge"] == "right_only"][["id_poliza", "nombre_asegurado_aseguradora"]].to_string(index=False))

## Nivel 3: Diferencias de prima y exportar

`fillna(0)` es solo para comparar: si falta la prima en una fuente, cuenta como diferencia. En el Excel la celda sigue vacía.

In [ ]:
d = m[(m["_merge"] == "both") &
      (m["prima_sistema"].fillna(0) != m["prima_aseguradora"].fillna(0))]
d = d[["id_poliza", "prima_sistema", "prima_aseguradora"]]

d.to_excel("diferencias_prima.xlsx", index=False)
d

In [ ]:
from google.colab import files
files.download("diferencias_prima.xlsx")